In [1]:
import os
import pandas as pd
import numpy  as np
from datetime import datetime

In [2]:
############################################ 
### import Principals filterd, > 4.5 GB
############################################
data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
import_file = os.path.join(data_path, "dwh_xref_titles_principals__filtered.csv")

P = pd.DataFrame()
P = pd.read_csv(import_file, encoding="utf-8")
 
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'P: ', P.shape)

2026-06-30 17:01:51 P:  (94787298, 6)


In [3]:
############################################ 
### import Principals details, > 1.5 GB
############################################
data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
import_file = os.path.join(data_path, "dwh_principals__details.csv")

N = pd.DataFrame()
N = pd.read_csv(import_file, encoding="utf-8")
 
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'N: ', N.shape)

2026-06-30 17:02:33 N:  (15448238, 8)


In [4]:
############################################ 
### import Titles enriched
############################################
data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
import_file = os.path.join(data_path, "dwh_titles__details.csv")

T = pd.DataFrame()
T = pd.read_csv(import_file, encoding="utf-8")
 
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'T: ', T.shape)

2026-06-30 17:02:37 T:  (637715, 12)


In [5]:
############################################ 
### import Titles enriched
############################################
data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
import_file = os.path.join(data_path, "dwh_titles__ratings.csv")

R = pd.DataFrame()
R = pd.read_csv(import_file, encoding="utf-8")
 
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'R: ', R.shape)

2026-06-30 17:02:38 R:  (1689173, 4)


In [6]:
############################################ 
### import my Ratings 
############################################ 
import os

data_path = r"G:/My Drive/casablanca/rpi/prod/interfaces/rpi10009_imdb__user_ratings/import/archive/"
import_file = os.path.join(data_path, "ratings.csv")

myR = pd.DataFrame()
myR = pd.read_csv(import_file, encoding="utf-8")
myR.rename(columns = {  "Your Rating": "tconst_your_rating"
                      , "Num Votes":   "tconst_your_num_votes"
                      , "Const": "tconst"}
                      , inplace = True)
myR.tail(3)
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'),'myR: ', myR.shape)



2026-06-30 17:02:38 myR:  (1124, 14)


In [7]:
#################################################
# Join
#################################################
from datetime import datetime
             
imdb = pd.merge(    P
                  , T 
                  , how = 'inner', left_on="tconst", right_on="tconst"
                 )
imdb = pd.merge(     N.loc[:, ('nconst','primaryProfession', 'primaryName', 'birthYear', 'deathYear', 'primaryName_year')]
                  , imdb
                  , how = 'inner', left_on="nconst", right_on="nconst"
                 )
imdb = pd.merge(     R.loc[:, ('tconst','tconst_imdb_rating', 'tconst_imdb_num_votes')]
                  , imdb
                  , how = 'inner', left_on="tconst", right_on="tconst"
                 )
imdb = pd.merge(    myR.loc[:, ('tconst','tconst_your_rating', 'Release Date' )]
                  , imdb
                  , how = 'right', left_on="tconst", right_on="tconst"
                 )
imdb['rate_diff']           = imdb['tconst_your_rating'] - imdb['tconst_imdb_rating']

imdb["age_crew"]            = np.where(imdb['birthYear']==0, ' ',  imdb['startYear'] - imdb['birthYear'])
imdb["last_refresh"]        = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
 
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'P: ', P.shape)
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'imdb: ', imdb.shape)

2026-06-30 17:05:43 P:  (94787298, 6)
2026-06-30 17:05:43 imdb:  (4665646, 29)


In [8]:
#imdb_dir = 
imdb_dir = imdb.loc[(imdb['cat_cln'] == "director") & (imdb['ordering'] == 0), ('tconst','primaryName')]  
imdb_dir.rename(columns = {  "primaryName": "primaryName_Dir0"}
                           , inplace = True)
imdb_dir.head(4)
imdb = pd.merge(    imdb_dir.loc[:, ('tconst','primaryName_Dir0')]
                  , imdb
                  , how = 'right', left_on="tconst", right_on="tconst"
                 )
 
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'imdb_dir: ', imdb_dir.shape)
print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'imdb: ', imdb.shape)
#imdb[imdb['tconst'] == "tt1745960"].head(15)
#iimdb[imdb['tconst'] == "tt0000335"].head(15)

2026-06-30 17:05:51 imdb_dir:  (342247, 2)
2026-06-30 17:05:51 imdb:  (4665646, 30)


In [9]:
#################################################
# add row number for filmmakers with more than one release per year
#################################################
imdb['RN'] = imdb.sort_values(['nconst', 'tconst'], ascending=[True, False]) \
             .groupby(['nconst', 'startYear']) \
             .cumcount() + 1
imdb.tail(10)

,tconst,primaryName_Dir0,tconst_your_rating,Release Date,tconst_imdb_rating,tconst_imdb_num_votes,nconst,primaryProfession,primaryName,birthYear,...,startYear,endYear,runtimeMinutes,genres,flg_doc,primaryTitle_year,rate_diff,age_crew,last_refresh,RN
4665636,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm10538614,producer,Ujjwala Gawde,0,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,,2026-06-30 17:05:43,1
4665637,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm13233318,"production_department,production_manager,actor",Ganesh Vasant Patil,0,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,,2026-06-30 17:05:43,1
4665638,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm1576284,actress,Amruta Subhash,1979,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,38,2026-06-30 17:05:43,1
4665639,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm1957275,"cinematographer,camera_department,producer",Suresh Deshmane,0,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,,2026-06-30 17:05:43,1
4665640,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm4289680,actor,Atul Todankar,0,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,,2026-06-30 17:05:43,1
4665641,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm4852679,actor,Bhushan Pradhan,0,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,,2026-06-30 17:05:43,1
4665642,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm6096005,actor,Devadhar Archit,0,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,,2026-06-30 17:05:43,1
4665643,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm7365126,actress,Aarti Solanki,0,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,,2026-06-30 17:05:43,1
4665644,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm9050497,actor,Pranav Raorane,0,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,,2026-06-30 17:05:43,1
4665645,tt9916730,Kiran Gawade,NaN,NaN,6.5,15,nm9785908,"assistant_director,editor,miscellaneous",Rohita More,0,...,2017,0,116,Drama,Fiction,6 Gunn (2017),NaN,,2026-06-30 17:05:43,1


In [10]:
# imdb --> input for tableau dashboard and further aggregation 
import os

#data_path = r"G:/My Drive/casablanca/rpi/prod/transformations/"
data_path = r"G:/My Drive/casablanca/rpi/prod/interfaces/rpi20003_twb__cast_and_crew/export/current/"
# output_file = os.path.join(data_path, "mrt_xref_titles_principals__enriched.csv")
output_file = os.path.join(data_path, "imdb.csv")
imdb.to_csv(output_file, encoding="utf-8")